# MedCoder-SLM-1B — Free Training on Google Colab T4

**What this does:** Fine-tunes Llama-3.2-1B-Instruct for medical coding (ICD-10 + CPT) using QLoRA.

**Requirements:**
- Free Colab T4 GPU (Runtime → Change runtime type → T4 GPU)
- HuggingFace account with access to Llama-3.2-1B-Instruct
- ~4 hours runtime

**Result:** A 1B model that assigns ICD-10 and CPT codes from clinical notes at ~69% exact match accuracy.

In [ ]:
# Step 1: Check GPU
!nvidia-smi
import torch
print(f'GPU available: {torch.cuda.is_available()}')
print(f'GPU name: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None"}')

In [ ]:
# Step 2: Install dependencies
!pip install -q transformers datasets peft trl bitsandbytes accelerate huggingface_hub
print('Dependencies installed')

In [ ]:
# Step 3: Login to HuggingFace
# Get your token at https://huggingface.co/settings/tokens (needs READ access to Llama)
from huggingface_hub import login
login()  # will prompt for token

In [ ]:
# Step 4: Clone training scripts
!git clone https://github.com/YOUR_USERNAME/medcoder-slm.git
%cd medcoder-slm

In [ ]:
# Step 5: Build dataset (~2 min)
!python build_dataset.py
!wc -l data/medcoder_train.jsonl data/medcoder_eval.jsonl

In [ ]:
# Step 6: Train (~4 hours on T4)
# Tip: Enable Colab Pro for faster A100 training (~1.5 hours)
!python train.py

In [ ]:
# Step 7: Evaluate
!python evaluate.py --model ./medcoder-slm-1b --n 100

In [ ]:
# Step 8: Quick inference test
import sys
sys.path.append('.')
from inference import MedCoder

mc = MedCoder('./medcoder-slm-1b')

note = """
72-year-old male with hypertension and T2DM presented with acute chest pain
radiating to left arm, diaphoresis x 2 hours. EKG: ST elevations V1-V4.
Troponin I 2.8 peaked at 18.4. Taken to cath lab, LAD 100% occlusion,
drug-eluting stent placed. Post-procedure EF 45%.
"""

result = mc.code(note)
import json
print(json.dumps(result, indent=2))

In [ ]:
# Step 9: Push to HuggingFace Hub
HF_USERNAME = 'your-hf-username'   # change this
HF_TOKEN    = 'your-write-token'   # from https://huggingface.co/settings/tokens

!python push_to_hub.py --hf-token {HF_TOKEN} --username {HF_USERNAME}

In [ ]:
# Done! Your model is now:
# 1. Live on HuggingFace: https://huggingface.co/YOUR_USERNAME/MedCoder-SLM-1B
# 2. The SQL INSERT for your SLM Marketplace was printed above — run it in Supabase.
print('Training complete!')
print(f'Model: https://huggingface.co/{HF_USERNAME}/MedCoder-SLM-1B')
print(f'Marketplace: https://slm-market.sannan.app/model?id=<your-model-id>')